# 01 — Three-Agent Dialogue (you are the Sam the customer)

A multi-agent conversation powered by the Mistral Python SDK. Three agents talk to each other in a loop —
and **you, the notebook user, play the first one**:

| # | Agent | Persona | Who speaks |
|---|-------|---------|------------|
| 1 | **Sam** | the customer — a tech-friendly person who wants to use AI to reflect on the world | **you**, via an `input()` prompt |
| 2 | **Vera** | a world-renowned, respected economist who knows how the world and money actually work | Mistral model |
| 3 | **Anton** | a famous philosopher who wrote and talked all his life about the good life, drawing on Aristotle, Stoicism and Karl Marx | Mistral model |

How it works:

- the dialogue runs in **3 turns**; each turn is one round: **Sam → Vera → Anton**;
- the **top of every loop iteration prompts you for Sam's comment** — on turn 1 this is the opening sentence that Vera, then Anton, comment on;
- the whole conversation is tracked in a **transcript** (who said what, in order);
- every utterance is capped at **5 sentences** (your input is truncated to 5; model replies are capped in the prompt and in code).

Setup: copy `.env.example` to `.env` at the project root and set `MISTRAL_API_KEY`.

In [ ]:
from mistral_jupyter import DEFAULT_MODEL, complete, get_client

client = get_client()
print("client ready — model:", DEFAULT_MODEL)

## 1. Personas and system prompts

Sam's persona — that is the role **you** play at the input prompts. Vera and Anton each get a system prompt: their persona, the cast of the dialogue, and the rules (read the labeled transcript, reply in at most 5 sentences, stay in character, engage with what the others said).

In [ ]:
SAM_PERSONA = """
You are Sam, a tech-friendly software engineer in your early 30s. You use AI tools every
day, and you want to use them to reflect on the world — to think out loud together with
sharper interlocutors, not just to get answers.
""".strip()

ECONOMIST_SYSTEM_PROMPT = """
You are Vera, a world-renowned and widely respected economist. You have spent your career
studying how the world actually works — incentives, markets, money, trade, institutions,
growth and inequality — and you explain it precisely, without jargon.

You are in a three-way written dialogue with:
- Sam, a tech-friendly customer who wants to use AI to reflect on the world
- Anton, a philosopher of the good life

Rules of the dialogue:
- You see the full transcript so far, with every line labeled by speaker name.
- When it is your turn, reply in at most 5 sentences.
- Stay in character: ground the conversation in how economies and money actually function.
  Use concrete mechanisms (incentives, prices, capital, debt, institutions), not slogans.
- Engage with what Sam and Anton actually said before making your point.
""".strip()

PHILOSOPHER_SYSTEM_PROMPT = """
You are Anton, a famous philosopher who has written and lectured all his life about the
good life. Your thinking weaves together three traditions:
- Aristotle: the good life (eudaimonia) is a life of virtue, cultivated as a habit, lived in a community.
- Stoicism: focus on what is up to you, accept what is not, and live according to reason.
- Karl Marx: under capitalism work can become alienated; a good life also means free, creative
  activity and solidarity, not just consumption.

You are in a three-way written dialogue with:
- Sam, a tech-friendly customer who wants to use AI to reflect on the world
- Vera, a world-renowned economist

Rules of the dialogue:
- You see the full transcript so far, with every line labeled by speaker name.
- When it is your turn, reply in at most 5 sentences.
- Stay in character: speak about the good life, virtue, freedom and meaning, drawing on
  Aristotle, the Stoics and Marx where they help.
- Engage with what Sam and Vera actually said before making your philosophical point.
""".strip()

print("Sam's persona — the role YOU play at the prompts:")
print(SAM_PERSONA)
for name, prompt in [
    ("Vera", ECONOMIST_SYSTEM_PROMPT),
    ("Anton", PHILOSOPHER_SYSTEM_PROMPT),
]:
    print(f"--- system prompt: {name} ({len(prompt)} chars) ---")

## 2. The agentic loop

Each **turn** is one round of the trio, and the **top of the loop always prompts you for Sam's comment**:

1. `input()` asks the notebook user (you) for Sam's line — on turn 1 this is the opening sentence of the dialogue; your input is truncated to `MAX_SENTENCES` sentences;
2. **Vera** (economist) comments — she sees her own system prompt plus the full labeled transcript;
3. **Anton** (philosopher) comments the same way.

Every utterance is appended to `transcript` (`{turn, speaker, persona, content}`), so the whole context — who said what — is tracked and fed back to the speakers. Model replies are capped at `MAX_SENTENCES = 5` sentences in the prompt and truncated in code as a safety net. The loop runs `N_TURNS = 3` rounds in total.

In [ ]:
import re
from dataclasses import dataclass

MAX_SENTENCES = 5  # hard cap per utterance
N_TURNS = 3  # number of rounds; each round = Sam (you) -> Vera -> Anton


@dataclass
class Agent:
    name: str
    persona: str
    system_prompt: str
    human: bool = False  # Sam is played by the notebook user, not by the model


agents = [
    Agent("Sam", "customer (you)", SAM_PERSONA, human=True),
    Agent("Vera", "economist", ECONOMIST_SYSTEM_PROMPT),
    Agent("Anton", "philosopher", PHILOSOPHER_SYSTEM_PROMPT),
]

# The transcript tracks the whole context of the conversation: who said what, in order.
transcript: list[dict] = []


def split_sentences(text: str) -> list[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]


def limit_sentences(text: str, max_sentences: int = MAX_SENTENCES) -> str:
    """Safety net: keep at most `max_sentences` sentences."""
    return " ".join(split_sentences(text)[:max_sentences])


def build_messages(
    speaker: Agent, transcript: list[dict], max_sentences: int = MAX_SENTENCES
) -> list[dict]:
    """System prompt of the current speaker + full labeled transcript + 'your turn'."""
    messages = [{"role": "system", "content": speaker.system_prompt}]
    for entry in transcript:
        role = "assistant" if entry["speaker"] == speaker.name else "user"
        messages.append(
            {"role": role, "content": f"{entry['speaker']}: {entry['content']}"}
        )
    messages.append(
        {
            "role": "user",
            "content": (
                "The labeled transcript above is the full conversation so far. "
                f"Your turn, {speaker.name} — reply in at most {max_sentences} sentences."
            ),
        }
    )
    return messages


def speak(agent: Agent, turn: int) -> str:
    """One model turn: build messages from the transcript, call Mistral, record the reply."""
    response = complete(
        build_messages(agent, transcript), client=client, max_tokens=512
    )
    reply = limit_sentences(response.choices[0].message.content)
    # drop a self-label if the model prefixes its own name (labels come from the transcript)
    reply = re.sub(rf"^{re.escape(agent.name)}\s*:\s*", "", reply).strip()
    transcript.append(
        {
            "turn": turn,
            "speaker": agent.name,
            "persona": agent.persona,
            "content": reply,
        }
    )
    print(
        f"Turn {turn} | {agent.name} ({agent.persona}) "
        f"| {len(split_sentences(reply))} sentences | {response.usage.total_tokens} tokens"
    )
    print(f"{agent.name}: {reply}\n")
    return reply


for turn in range(1, N_TURNS + 1):
    print(f"\n===== Turn {turn}/{N_TURNS} ======")
    if turn == 1:
        print("Remember your persona as Sam:\n" + SAM_PERSONA + "\n")
    # Top of the loop: the notebook user is Sam — prompt for his comment.
    sam_line = input(f"Sam (you), your comment — at most {MAX_SENTENCES} sentences: ")
    sam_line = limit_sentences(sam_line)
    transcript.append(
        {
            "turn": turn,
            "speaker": "Sam",
            "persona": "customer (you)",
            "content": sam_line,
        }
    )
    print(
        f"Turn {turn} | Sam (customer, you) | {len(split_sentences(sam_line))} sentence(s)"
    )
    print(f"Sam: {sam_line}\n")

    # Then Vera, then Anton comment on the conversation so far.
    for agent in agents[1:]:
        speak(agent, turn)

## 3. The full transcript — who said what

Recap of the whole conversation from the transcript, plus checks: exactly `N_TURNS` rounds, each round in the order Sam → Vera → Anton, and every utterance within the 5-sentence cap.

In [ ]:
for entry in transcript:
    n = len(split_sentences(entry["content"]))
    print(
        f"Turn {entry['turn']} — {entry['speaker']} ({entry['persona']}), {n} sentence(s):"
    )
    print(entry["content"])
    print()

rounds = sorted({e["turn"] for e in transcript})
assert rounds == list(range(1, N_TURNS + 1)), (
    "the dialogue must have exactly N_TURNS rounds"
)
assert all(len(split_sentences(e["content"])) <= MAX_SENTENCES for e in transcript), (
    "5-sentence cap violated"
)
for t in rounds:
    order = [e["speaker"] for e in transcript if e["turn"] == t]
    assert order == ["Sam", "Vera", "Anton"], f"round {t} order wrong: {order}"
print(
    f"OK: {len(rounds)} turns, each Sam -> Vera -> Anton, every utterance <= {MAX_SENTENCES} sentences"
)